In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.set_option('display.max_rows', None)

In [16]:
df = pd.read_json("/home/user/part_01.json",  lines=True)
df.head()   

,id,agent_name,amenities,bathrooms,bedrooms,broker,broker_display_name,category,category_url,completion_status,...,reference_number,rera_permit_number,rera_registration_number,scraped_ts,sub_category_1,sub_category_2,title,url,user_id,verified
0,26296235,Varun Arora,"Central A/C & Heating, Balcony, Private Garden...",5,5,KAVANI HOMEZ REAL ESTATE,Kavani Homez Real Estate,rent,/property-for-rent/home/,,...,10865-94B9Fr,65382196505,27726,2026-06-22,residential,,5 BR Townhouse l Corner Unit l Brand New,https://dubai.dubizzle.com/property-for-rent/r...,,Checked
1,27075435,Imtiaz Ahmed,"Central A/C & Heating, Concierge Service, Priv...",2,1,BELLA CASA REAL ESTATE,Bella Casa Real Estate,rent,/property-for-rent/home/,,...,9948-xHRRX3,7127094200,17382,2026-06-22,residential,,Spacious Furnished 1BR | High-Floor Marina Living,https://dubai.dubizzle.com/property-for-rent/r...,,TruCheckTM
2,27238755,Fahd Hossam Abdelghany Senedak,"Central A/C & Heating, Balcony, Kids Play Area...",2,1,COLDWELL BANKER - SWAP REAL ESTATE,Coldwell Banker - Swap Real Estate,rent,/property-for-rent/home/,,...,CB-R-07525,7146458757,24905,2026-06-22,residential,,Ready To Move | Fully Furnished | High Floor,https://dubai.dubizzle.com/property-for-rent/r...,,Checked
3,26528203,Mohamad Nabizadeh,"Central A/C & Heating, Concierge Service, Balc...",3,2,PROPERTYSHOMA REAL ESTATE,Propertyshoma Real Estate,rent,/property-for-rent/home/,,...,10499-Psaqw8,7195867750,26287,2026-06-22,residential,,Fully Furnished & Upgraded 2BR for Rent | Hear...,https://dubai.dubizzle.com/property-for-rent/r...,,Checked
4,27467677,Chaima Adami,"Central A/C & Heating, Balcony",5,4,COLDWELL BANKER - SWAP REAL ESTATE,Coldwell Banker - Swap Real Estate,rent,/property-for-rent/home/,,...,CB-R-07757,65387169943,24905,2026-06-22,residential,,Brand New | Corner Unit | Lagoon View,https://dubai.dubizzle.com/property-for-rent/r...,,


In [4]:
df.columns

Index(['id', 'agent_name', 'amenities', 'bathrooms', 'bedrooms', 'broker',
       'broker_display_name', 'category', 'category_url', 'completion_status',
       'currency', 'date', 'ded_license_number', 'depth', 'description',
       'details', 'dtcm_licence', 'furnished', 'iteration_number',
       'last_update', 'latitude', 'listed_by', 'location', 'longitude',
       'number_of_photos', 'package_type', 'phone_number', 'price',
       'price_per', 'property_type', 'published_at', 'reference_number',
       'rera_permit_number', 'rera_registration_number', 'scraped_ts',
       'sub_category_1', 'sub_category_2', 'title', 'url', 'user_id',
       'verified'],
      dtype='object')

In [11]:

df.id.value_counts()

id
27101895    1
27163342    1
27104429    1
27153782    1
27163555    1
27462999    1
27142829    1
27463003    1
27463002    1
27156089    1
27160722    1
27463004    1
27161616    1
27463011    1
27161619    1
27463007    1
27476402    1
27474396    1
27342934    1
27449432    1
27476317    1
27472815    1
27473755    1
27444369    1
27430697    1
27244063    1
27476010    1
27433176    1
27151123    1
27398636    1
27394516    1
26903374    1
27473408    1
27224433    1
27337766    1
27475767    1
27344649    1
27463156    1
27475119    1
27447299    1
27415652    1
27462593    1
27444546    1
26644225    1
27381126    1
27462763    1
27082786    1
27477021    1
27416279    1
27481003    1
27473400    1
26963258    1
27353272    1
26841465    1
27398394    1
27079433    1
27459907    1
27352594    1
27130212    1
27386086    1
27398395    1
27337398    1
27462138    1
27417205    1
27223527    1
26524915    1
27468669    1
27452316    1
27384176    1
27463238    1
27480326    1
274

In [12]:
mismatch = df[
    df["broker"].fillna("").str.strip().str.lower() !=
    df["broker_display_name"].fillna("").str.strip().str.lower()
]

print(f"Mismatched rows: {len(mismatch)}")
print(mismatch[["broker", "broker_display_name"]])

Mismatched rows: 0
Empty DataFrame
Columns: [broker, broker_display_name]
Index: []


In [13]:
df.iteration_number.value_counts()


iteration_number
20260604    100000
Name: count, dtype: int64

In [14]:
df = df.replace(r'^\s*$', pd.NA, regex=True)

# Find columns that are completely empty in all records
empty_columns = []

for col in df.columns:
    if df[col].notna().sum() == 0:
        empty_columns.append(col)

print("Completely empty columns:")
print(empty_columns)


Completely empty columns:
['ded_license_number', 'dtcm_licence', 'listed_by', 'user_id']


In [15]:
df.price_per.value_counts()


price_per
Yearly       77271
Monthly      14746
Quarterly     1441
Bi-Yearly       17
Name: count, dtype: int64

In [40]:
import re


pattern = re.compile(
    r'^https?:\/\/([A-Za-z0-9-]+\.)+[A-Za-z]{2,}(\/[^\s]*)?$'
)

invalid_urls = df[
    ~df["url"].fillna("").astype(str).str.match(pattern)
]

print(invalid_urls["url"])

Series([], Name: url, dtype: object)


In [10]:
import json

fields = set()
records = []

with open("/home/user/part_01.json", "r", encoding="utf-8") as f:
    for line in f:
        record = json.loads(line)
        records.append(record)
        fields.update(record.keys())

empty_fields = []

for field in fields:
    if all(record.get(field) in [None, "", [], {}] for record in records):
        empty_fields.append(field)

print("Completely empty fields:")
for field in sorted(empty_fields):
    print(field)

Completely empty fields:
ded_license_number
dtcm_licence
listed_by
user_id
